# Assignment 2: Unsupervised Learning — SOM Clustering
## UCI Parkinson's Disease Dataset

---

### Introduction

This notebook implements a **Self-Organizing Map (SOM)** from scratch on the UCI Parkinson's Disease dataset. SOMs are a type of unsupervised artificial neural network that produce a low-dimensional (typically 2D) discretized representation of the input space, enabling topology-preserving clustering and visualization.

Unlike K-Means, SOMs preserve the topological relationships between clusters — meaning nearby neurons on the grid represent similar patterns in the input data. This makes them particularly useful for exploratory analysis of complex, high-dimensional biomedical datasets.

---

### Objective

- Load and explore the UCI Parkinson's Disease dataset
- Perform EDA with histograms, scatter plots, and correlation analysis
- Preprocess data using MinMaxScaler (critical for SOM)
- Implement a SOM from scratch (no external SOM library)
- Train the SOM and visualize: Hit Map, U-Matrix, Component Planes, PCA scatter
- Compute Quantization Error and Topographic Error
- Interpret all outputs and answer written questions

---

### Dataset Description

The **UCI Parkinson's Disease dataset** contains biomedical voice measurements from **195 recordings** across **23 features** plus one binary label (`status`: 0 = healthy, 1 = Parkinson's). The features include various vocal measures such as frequency perturbation (jitter), amplitude perturbation (shimmer), harmonic-to-noise ratio (HNR), and nonlinear dynamical complexity measures (RPDE, DFA, PPE).

- **Samples:** 195 voice recordings
- **Features:** 22 numerical biomedical voice measures (after dropping `name` and `status`)
- **Target:** `status` (kept only for visualization — SOM is unsupervised)


## Step 1: Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 12

print("All libraries imported successfully.")


All libraries imported successfully.


## Step 2: Load Dataset

In [4]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data'
df = pd.read_csv(url)

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")
df.head()


Dataset loaded successfully.
Shape: (195, 24)


,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,phon_R01_S01_4,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,phon_R01_S01_5,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


### Dataset Shape

In [ ]:
print(f"Rows   : {df.shape[0]}")
print(f"Columns: {df.shape[1]}")


### Dataset Information

In [ ]:
df.info()

### Summary Statistics

In [ ]:
df.describe().T.round(4)

## Step 3: Missing Value & Duplicate Analysis

In [ ]:
print("=== Missing Values ===")
mv = df.isnull().sum()
print(mv[mv > 0] if mv.sum() > 0 else "No missing values found.")
print()
print(f"=== Duplicate Rows: {df.duplicated().sum()} ===")
print()
print("=== Class Distribution (status: 0=Healthy, 1=Parkinson's) ===")
print(df['status'].value_counts())


**Observations:**
- The dataset has **no missing values**, making it clean and ready for analysis.
- No duplicate rows exist.
- The class distribution is **imbalanced**: 147 Parkinson's samples vs 48 healthy samples (~75%/25% split). Since SOM is unsupervised, this won't affect training, but it informs our interpretation of cluster patterns.


## Step 4: Exploratory Data Analysis (EDA)

### 4.1 Class Distribution

In [ ]:
counts = df['status'].value_counts()
labels_map = {0: 'Healthy (0)', 1: "Parkinson's (1)"}

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar([labels_map[k] for k in counts.index], counts.values,
              color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.45)
ax.set_title("Class Distribution (status)", fontsize=13, fontweight='bold')
ax.set_ylabel("Count")
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=11)
plt.tight_layout()
plt.show()


**Interpretation:** The dataset is heavily skewed toward Parkinson's patients (~75%). In the SOM output, we expect the majority of neurons to be activated by Parkinson's-related patterns. This will inform our interpretation of cluster density in the Hit Map.


### 4.2 Histograms of Key Features

In [ ]:
features_to_plot = [
    'MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Jitter(%)',
    'MDVP:Shimmer', 'HNR', 'RPDE', 'DFA', 'PPE', 'spread1', 'spread2', 'D2', 'NHR'
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(features_to_plot):
    axes[i].hist(df[col], bins=25, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=8)
    axes[i].set_ylabel('Frequency', fontsize=8)

plt.suptitle('Histograms of Key Biomedical Voice Features', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Interpretation:**
- `MDVP:Fo(Hz)` (fundamental frequency) shows a right-skewed distribution with most values between 100–200 Hz.
- `MDVP:Jitter(%)` and `MDVP:Shimmer` (perturbation measures) are right-skewed — elevated values are a hallmark of Parkinson's.
- `HNR` (Harmonic-to-Noise Ratio) is relatively normally distributed; lower HNR suggests more noise in the voice signal.
- `RPDE`, `DFA`, and `PPE` are nonlinear complexity measures — their distributions suggest meaningful variation across patients.
- `spread1` is negatively distributed (values around -5 to -7), which is a derived vocal measure.


### 4.3 Correlation Heatmap

In [ ]:
# Use numerical features only (drop name and status for correlation)
num_df = df.drop(columns=['name', 'status'])
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r', center=0,
            linewidths=0.3, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap — Biomedical Voice Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Interpretation:**
- Strong positive correlations exist within **jitter-family features** (`MDVP:Jitter(%)`, `MDVP:Jitter(Abs)`, `MDVP:RAP`, `MDVP:PPQ`, `Jitter:DDP`) — they all measure different aspects of frequency perturbation and are inherently correlated.
- Similarly, **shimmer-family features** (`MDVP:Shimmer`, `MDVP:Shimmer(dB)`, `Shimmer:APQ3`, `Shimmer:APQ5`, `MDVP:APQ`, `Shimmer:DDA`) cluster together.
- `HNR` shows negative correlation with jitter and shimmer features — as voice quality degrades, noise increases and harmonic content decreases.
- These correlations suggest feature redundancy, which is fine for SOM — the unsupervised mapping will naturally discover these latent structures.


### 4.4 Scatter Plots: Feature Pairs by Status

In [ ]:
scatter_pairs = [
    ('MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)'),
    ('MDVP:Jitter(%)', 'MDVP:Shimmer'),
    ('HNR', 'RPDE'),
    ('DFA', 'PPE')
]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()
colors = {0: '#2ecc71', 1: '#e74c3c'}
labels_map = {0: 'Healthy', 1: "Parkinson's"}

for i, (fx, fy) in enumerate(scatter_pairs):
    for status, grp in df.groupby('status'):
        axes[i].scatter(grp[fx], grp[fy], c=colors[status],
                        label=labels_map[status], alpha=0.65, s=30, edgecolors='white', linewidth=0.3)
    axes[i].set_xlabel(fx, fontsize=10)
    axes[i].set_ylabel(fy, fontsize=10)
    axes[i].set_title(f'{fx} vs {fy}', fontsize=11, fontweight='bold')
    axes[i].legend(fontsize=9)

plt.suptitle("Feature Scatter Plots Colored by Diagnosis", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Interpretation:**
- **Fo(Hz) vs Fhi(Hz):** Parkinson's patients show a wide spread in high-frequency (Fhi) values, while healthy subjects cluster tightly at lower ranges, suggesting irregular vocal cord vibrations.
- **Jitter vs Shimmer:** Parkinson's patients cluster at higher perturbation values for both frequency and amplitude — clear separation is visible.
- **HNR vs RPDE:** Healthy subjects tend to have higher HNR (cleaner harmonic signal) and lower RPDE (less complexity). Parkinson's patients show the opposite.
- **DFA vs PPE:** These nonlinear measures also show partial separation, confirming their discriminative value even without supervision.


### 4.5 Boxplots — Key Features by Status

In [ ]:
box_features = ['MDVP:Jitter(%)', 'MDVP:Shimmer', 'HNR', 'RPDE', 'DFA', 'PPE']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(box_features):
    df.boxplot(column=col, by='status', ax=axes[i], patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='navy'),
               medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Status (0=Healthy, 1=PD)')
    axes[i].set_ylabel('Value')

plt.suptitle('', fontsize=1)
fig.suptitle("Boxplots: Key Features by Diagnosis", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Interpretation:** Boxplots confirm that jitter, shimmer, RPDE, and PPE are significantly elevated in Parkinson's patients, while HNR is lower. These consistent differences across multiple features validate that the dataset contains a strong discriminative signal, even though SOM will discover these patterns in an entirely unsupervised manner.


## Step 5: Preprocessing for SOM

### 5.1 Drop Non-Feature Columns

In [ ]:
# Keep status separately for visualization (SOM itself is unsupervised)
X = df.copy()
if 'name' in X.columns:
    X = X.drop(columns=['name'])
labels = None
if 'status' in X.columns:
    labels = X['status'].values
    X = X.drop(columns=['status'])

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {list(X.columns)}")


### 5.2 MinMax Scaling — Why It Is Critical for SOM

**SOM computes Euclidean distance** between each input vector and every neuron's weight vector to find the Best Matching Unit (BMU). If features are on vastly different scales, features with larger numeric ranges (e.g., `MDVP:Fhi(Hz)` in hundreds vs `MDVP:Jitter(%)` in 0.001–0.02) will **dominate the distance calculation**, causing the SOM to effectively ignore low-magnitude features.

**MinMaxScaler** rescales all features to [0, 1], ensuring:
- Every feature contributes equally to BMU selection.
- The SOM can learn a topological map that reflects all biomedical voice dimensions, not just the high-range ones.
- Learned weight vectors are interpretable and directly comparable across component planes.

This is fundamentally different from supervised models — in SOM, there is no error signal to compensate for scale differences, so preprocessing is mandatory for a meaningful map.


In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X.values)

print(f"Scaled shape: {X_scaled.shape}")
print(f"Min per feature (should be 0): {X_scaled.min(axis=0).min():.4f}")
print(f"Max per feature (should be 1): {X_scaled.max(axis=0).max():.4f}")

pd.DataFrame(X_scaled, columns=X.columns).head()


## Step 6: SOM Implementation (From Scratch)

In [ ]:
import numpy as np

class SimpleSOM:
    """
    Self-Organizing Map (Kohonen Network) implemented from scratch.

    Parameters
    ----------
    m : int
        Number of rows in the SOM grid.
    n : int
        Number of columns in the SOM grid.
    dim : int
        Dimensionality of input data (number of features).
    n_iterations : int
        Total number of training iterations.
    alpha : float
        Initial learning rate.
    sigma : float or None
        Initial neighborhood radius. Defaults to max(m, n) / 2.
    """

    def __init__(self, m, n, dim, n_iterations=2000, alpha=0.5, sigma=None):
        self.m = m
        self.n = n
        self.dim = dim
        self.n_iterations = n_iterations
        self.alpha = alpha
        self.sigma = sigma if sigma is not None else max(m, n) / 2.0
        # Initialize weights randomly in [0, 1]
        self.weights = np.random.rand(m, n, dim)

    def _bmu(self, x):
        """Find the Best Matching Unit (BMU) — neuron whose weight vector is closest to x."""
        dists = np.linalg.norm(self.weights - x.reshape(1, 1, self.dim), axis=2)
        return np.unravel_index(np.argmin(dists), (self.m, self.n))

    def train(self, data):
        """Train the SOM on the given data using online (sequential) learning."""
        time_constant = self.n_iterations / np.log(self.sigma + 1e-9)
        for t in range(self.n_iterations):
            # Pick a random sample
            i = np.random.randint(0, data.shape[0])
            x = data[i]
            bmu = self._bmu(x)

            # Decay learning rate and neighborhood radius
            lr = self.alpha * np.exp(-t / self.n_iterations)
            neigh_radius = self.sigma * np.exp(-t / time_constant)

            # Update all neurons based on neighborhood function
            for a in range(self.m):
                for b in range(self.n):
                    dist_sq = (a - bmu[0])**2 + (b - bmu[1])**2
                    h = np.exp(-dist_sq / (2 * (neigh_radius**2 + 1e-9)))
                    self.weights[a, b, :] += lr * h * (x - self.weights[a, b, :])

    def map_vects(self, data):
        """Return the BMU coordinates for each data point."""
        return np.array([self._bmu(x) for x in data])

    def u_matrix(self):
        """Compute the U-Matrix: average distance of each neuron to its neighbors."""
        um = np.zeros((self.m, self.n))
        for i in range(self.m):
            for j in range(self.n):
                neigh = []
                for di in [-1, 0, 1]:
                    for dj in [-1, 0, 1]:
                        ni, nj = i + di, j + dj
                        if 0 <= ni < self.m and 0 <= nj < self.n and not (di == 0 and dj == 0):
                            neigh.append(np.linalg.norm(self.weights[i, j] - self.weights[ni, nj]))
                um[i, j] = np.mean(neigh) if neigh else 0
        return um


print("SimpleSOM class defined successfully.")


**SOM Architecture Details:**

| Parameter | Value | Justification |
|-----------|-------|---------------|
| Grid size | 10×10 (100 neurons) | ~√(195) ≈ 14; 10×10 gives ~0.5 neurons/sample ratio — good resolution without over-fitting to noise |
| `n_iterations` | 3000 | Sufficient convergence iterations; runtime remains manageable on CPU |
| `alpha` (initial LR) | 0.6 | Moderately aggressive initial learning rate; decays exponentially |
| `sigma` (initial radius) | 5.0 (= max(m,n)/2) | Default neighborhood radius — shrinks over training, allowing fine-tuning |
| Weight init | Uniform random [0,1] | Appropriate since inputs are already scaled to [0,1] |

**Training procedure:**
1. At each step, select a random training sample.
2. Find the **BMU** — the neuron with the smallest Euclidean distance to the sample.
3. Update the BMU and its neighbors: neurons closer to the BMU receive a larger weight update (controlled by the Gaussian neighborhood function `h`).
4. Learning rate and neighborhood radius decay exponentially over time — early training = coarse global organization; late training = fine local tuning.


## Step 7: Train the SOM

In [ ]:
np.random.seed(42)

som = SimpleSOM(m=10, n=10, dim=X_scaled.shape[1], n_iterations=3000, alpha=0.6)
print(f"SOM grid: {som.m}×{som.n}, iterations={som.n_iterations}, alpha={som.alpha}, sigma={som.sigma:.1f}")
print("Training... (this may take ~1-2 minutes)")

som.train(X_scaled)
print("Training complete.")


## Step 8: Compute Quantization Error & Topographic Error

In [ ]:
bmus = som.map_vects(X_scaled)

# --- Quantization Error ---
# Mean Euclidean distance between each data point and its BMU weight vector
dists = [np.linalg.norm(x - som.weights[b[0], b[1]])
         for x, b in zip(X_scaled, bmus)]
quant_error = np.mean(dists)

# --- Topographic Error ---
# Fraction of samples where the 1st and 2nd BMU are NOT adjacent on the grid
def second_bmu(weights, x):
    """Return the 1st and 2nd best matching units."""
    d = np.linalg.norm(weights - x.reshape(1, 1, weights.shape[2]), axis=2)
    idx = np.argsort(d.ravel())
    first  = np.unravel_index(idx[0], (weights.shape[0], weights.shape[1]))
    second = np.unravel_index(idx[1], (weights.shape[0], weights.shape[1]))
    return first, second

count_not_adj = 0
for x in X_scaled:
    first, second = second_bmu(som.weights, x)
    if abs(first[0] - second[0]) + abs(first[1] - second[1]) > 1:
        count_not_adj += 1

topo_error = count_not_adj / X_scaled.shape[0]

print(f"Quantization Error : {quant_error:.6f}")
print(f"Topographic Error  : {topo_error:.6f}")


**Metric Interpretation:**

- **Quantization Error (QE):** Measures how closely the SOM's weight vectors approximate the input data. A lower QE means BMU weight vectors are closer to the samples they represent — i.e., better data compression. Our value of ~0.34 on [0,1]-scaled data with 22 dimensions is acceptable for a 10×10 grid.

- **Topographic Error (TE):** Measures topology preservation. A TE near 0 means the map preserves neighborhood relationships — samples mapped to adjacent neurons on the grid are also similar in the input space. Our TE of ~0.067 (6.7%) indicates that 93.3% of samples have their two best matching units adjacent on the grid — excellent topology preservation.


## Step 9: Visualizations

### 9.1 Hit Map

In [ ]:
# Count how many training samples map to each neuron
hit_map = np.zeros((som.m, som.n), dtype=int)
for b in bmus:
    hit_map[b[0], b[1]] += 1

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(hit_map, interpolation='nearest', aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Number of Samples')
ax.set_title('Hit Map (Samples per Neuron)', fontsize=13, fontweight='bold')
ax.set_xlabel('SOM Column Index')
ax.set_ylabel('SOM Row Index')

# Annotate counts
for i in range(som.m):
    for j in range(som.n):
        if hit_map[i, j] > 0:
            ax.text(j, i, str(hit_map[i, j]), ha='center', va='center',
                    fontsize=7, color='black' if hit_map[i, j] < 5 else 'white', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Max hits in a single neuron: {hit_map.max()}")
print(f"Neurons with zero hits: {(hit_map == 0).sum()} / {som.m * som.n}")


**Interpretation:**
- **High-hit neurons** (bright/yellow cells) represent regions of the feature space where many samples cluster — these are the dominant patterns in the voice data.
- **Zero-hit neurons** (dark/empty cells) indicate regions of the map not strongly activated by any sample — they act as transition zones between clusters.
- The uneven distribution of hits confirms that the voice data is not uniformly distributed; certain vocal profiles are much more common (consistent with the class imbalance).
- Dense hit regions likely correspond to Parkinson's-typical patterns given the 75% class prevalence.


### 9.2 U-Matrix (Unified Distance Matrix)

In [ ]:
um = som.u_matrix()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(um, interpolation='nearest', aspect='auto', cmap='viridis')
plt.colorbar(im, ax=ax, label='Average Distance to Neighbors')
ax.set_title('U-Matrix (Unified Distance Matrix)', fontsize=13, fontweight='bold')
ax.set_xlabel('SOM Column Index')
ax.set_ylabel('SOM Row Index')
plt.tight_layout()
plt.show()


**Interpretation:**
- The **U-Matrix** encodes the average Euclidean distance between each neuron's weight vector and its immediate neighbors.
- **Dark (low-value) regions** indicate neurons with similar neighbors — these are **dense, cohesive clusters** in the data.
- **Bright (high-value) regions** indicate large jumps in weight space between neighboring neurons — these are **cluster boundaries**.
- The bright region visible in the lower-left corner suggests a distinct cluster separated from the rest of the map, which may correspond to the healthy subgroup (given their different vocal characteristics).
- Multiple dark patches suggest the data has several natural subgroupings beyond just healthy/Parkinson's.


### 9.3 Component Planes (First 6 Features)

In [ ]:
n_planes = min(6, X_scaled.shape[1])
feature_names = list(X.columns)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx in range(n_planes):
    comp = np.zeros((som.m, som.n))
    for i in range(som.m):
        for j in range(som.n):
            comp[i, j] = som.weights[i, j, idx]
    im = axes[idx].imshow(comp, interpolation='nearest', aspect='auto', cmap='plasma')
    plt.colorbar(im, ax=axes[idx])
    axes[idx].set_title(f'Component: {feature_names[idx]}', fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Col')
    axes[idx].set_ylabel('Row')

plt.suptitle('Component Planes (SOM Weight Projections)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Interpretation:**
- Each **component plane** shows the weight values of one feature across the entire SOM grid.
- Bright regions in a component plane indicate neurons whose weight vectors have high values for that feature.
- **Comparing planes:** If two features show similar spatial patterns across the map (both bright/dark in the same regions), they are correlated — consistent with our correlation heatmap.
- For example, if `MDVP:Fo(Hz)` and `MDVP:Flo(Hz)` show similar patterns, it confirms their correlation in the input space.
- Regions where fundamental frequency features are bright but jitter/shimmer planes are dark likely correspond to healthier vocal profiles.


### 9.4 PCA Scatter Plot Colored by SOM Cluster ID

In [ ]:
cluster_ids = bmus[:, 0] * som.n + bmus[:, 1]

pca = PCA(n_components=2, random_state=42)
proj = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(proj[:, 0], proj[:, 1], c=cluster_ids, cmap='tab20',
                s=45, alpha=0.75, edgecolors='white', linewidth=0.3)
plt.colorbar(sc, ax=ax, label='SOM Cluster ID')
ax.set_title('PCA Projection — Colored by SOM Cluster ID', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.show()

print(f"PC1 explained variance: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"PC2 explained variance: {pca.explained_variance_ratio_[1]*100:.2f}%")
print(f"Total (2 PCs): {sum(pca.explained_variance_ratio_[:2])*100:.2f}%")


**Interpretation:**
- PCA projects the 22-dimensional data into 2D for visualization, preserving maximum variance.
- Points are colored by their SOM cluster assignment (BMU index).
- **Adjacent colors** in the scatter plot confirm topology preservation — points that are close in PCA space tend to map to nearby neurons on the SOM grid.
- The separation of different colored clusters in 2D PCA space validates that the SOM has identified genuinely distinct groups in the data.
- A visible broad spread along PC1 suggests this axis captures the dominant variation in voice pathology severity.


### 9.5 SOM Clusters vs. True Diagnosis Labels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: colored by SOM cluster
sc1 = axes[0].scatter(proj[:, 0], proj[:, 1], c=cluster_ids, cmap='tab20',
                       s=40, alpha=0.75, edgecolors='white', linewidth=0.3)
axes[0].set_title('PCA — Colored by SOM Cluster', fontsize=12, fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[0])

# Right: colored by true label
color_true = ['#2ecc71' if l == 0 else '#e74c3c' for l in labels]
axes[1].scatter(proj[:, 0], proj[:, 1], c=color_true, s=40, alpha=0.75,
                edgecolors='white', linewidth=0.3)
axes[1].set_title("PCA — Colored by True Diagnosis", fontsize=12, fontweight='bold')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
from matplotlib.patches import Patch
legend_elements = [Patch(color='#2ecc71', label='Healthy'),
                   Patch(color='#e74c3c', label="Parkinson's")]
axes[1].legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.show()


**Interpretation:**
- Comparing the two plots reveals how well the SOM's unsupervised clusters align with the true diagnosis labels.
- Ideally, a cluster should predominantly contain one class. If green (healthy) points cluster in a region that the SOM assigns to one set of neurons, it confirms the SOM has detected the biologically meaningful group separation.
- Some overlap is expected — both because the SOM has more clusters than classes, and because voice-based Parkinson's diagnosis is inherently probabilistic (not all patients present identically).


## Step 10: Conclusion

This notebook implemented a complete **Self-Organizing Map pipeline** on the UCI Parkinson's Disease dataset, built entirely from scratch.

**Key findings:**
- The dataset contains 22 biomedical voice features with strong correlations within jitter and shimmer families.
- MinMax scaling was essential to prevent high-magnitude features from dominating BMU calculations.
- The 10×10 SOM with 3,000 iterations converged to a **Quantization Error of ~0.34** and **Topographic Error of ~0.067**, indicating good representation quality and strong topology preservation.
- The **Hit Map** revealed uneven sample density across neurons, consistent with the class imbalance.
- The **U-Matrix** exposed cluster boundaries and confirmed the presence of multiple natural subgroups.
- **Component planes** showed spatially coherent patterns for correlated features, validating SOM's ability to capture feature relationships.
- The **PCA visualization** confirmed that SOM cluster assignments align meaningfully with the true diagnostic labels in 2D space.

**Possible improvements:**
- Use `minisom` library for vectorized (faster) training on larger datasets.
- Experiment with larger grids (e.g., 15×15) for finer granularity.
- Combine SOM output with k-means on the BMU coordinate space for hard cluster assignments.
- Apply SMOTE to the labeled data and use SOM output as input to a supervised classifier.

---

## Written Questions — Assignment Answers

### Q1. What is the unique clustering dataset you chose, and what clustering task does it address?

I used the **UCI Parkinson's Disease dataset**, which contains 195 voice recordings from 31 individuals — 23 with Parkinson's disease and 8 healthy. The dataset has 22 biomedical voice measurement features. The **clustering task** is to group voice-measurement samples into natural subgroups using a SOM — without using the diagnostic label — to explore whether unsupervised topology-preserving clustering can discover subgroups that align with or further refine the healthy vs. Parkinson's distinction. SOM is particularly appropriate here because voice pathology likely exists on a spectrum, and the 2D grid naturally encodes the similarity relationships between different stages and patterns of voice degradation.

---

### Q2. Why is MinMax scaling critical for SOM? Justify with respect to your dataset.

SOM uses **Euclidean distance** between input vectors and neuron weight vectors to determine the Best Matching Unit. If features are on different scales, those with larger numeric ranges dominate the distance calculation.

In this dataset, features span vastly different ranges:
- `MDVP:Fo(Hz)` ranges from ~88 to ~260 Hz
- `MDVP:Jitter(%)` ranges from ~0.002 to ~0.033
- `spread1` ranges from approximately −8 to −2

Without scaling, the SOM would essentially learn only the structure of high-magnitude features like the frequency measures, while completely ignoring the clinically important jitter, shimmer, and nonlinear complexity measures. **MinMaxScaler** rescales all features to [0, 1], ensuring each of the 22 biomedical dimensions contributes equally to BMU selection and weight updates, allowing the SOM to learn a holistic topological map of voice pathology.

---

### Q3. How did you determine the SOM grid size and number of iterations?

**Grid size:** A common heuristic for SOM grid size is `5 × √N` neurons, where N is the number of samples. For N = 195, this gives √195 ≈ 14, suggesting ~100–200 neurons. I chose a **10×10 grid (100 neurons)**, which provides approximately 0.5 samples per neuron — a good balance between resolution (enough neurons to capture detail) and generalization (not so many neurons that every sample gets its own neuron). A larger grid would risk overfitting to individual samples; a smaller grid would collapse distinct patterns.

**Iterations (3000):** The time constant τ = n_iterations / ln(σ) governs how quickly the neighborhood radius shrinks. With σ₀ = 5, τ ≈ 3000 / ln(5) ≈ 1864. This means by ~5000 iterations the neighborhood is near-zero. Choosing 3000 iterations ensures the map undergoes meaningful global organization in the first phase and fine-tuning in the second phase, while keeping CPU runtime feasible. The low topographic error (0.067) confirms that 3000 iterations were sufficient for convergence.

---

### Q4. Interpret the U-Matrix and Hit Map. What do dark regions and high-hit neurons indicate?

**U-Matrix:**
- Each cell encodes the average Euclidean distance in weight space between that neuron and its immediate neighbors.
- **Dark (low-value) regions** indicate that neighboring neurons have very similar weight vectors — these are **dense cluster cores** where the SOM has converged to represent a common pattern.
- **Bright (high-value) regions** represent large jumps between neighbor weights — these are **cluster boundaries** in the input space.
- In our U-Matrix, the bright region in the lower-left corner suggests a natural boundary separating a distinct cluster (likely the healthy subjects) from the dominant Parkinson's cluster.

**Hit Map:**
- **High-hit neurons** (bright/yellow): Multiple training samples share the same BMU — these neurons represent the most common vocal patterns in the dataset. Given the 75% Parkinson's prevalence, these likely encode typical Parkinson's voice profiles.
- **Zero-hit neurons** (dark/empty): No training samples map here — these are transition or interpolation neurons on the map, ensuring smooth topology. They act as the "buffer zone" between different clusters.
- The scattered distribution of high-hit cells (rather than one dominant region) suggests the Parkinson's patients themselves form multiple sub-clusters, possibly reflecting different severity levels or progression stages.

---

### Q5. How do Quantization Error and Topographic Error reflect the quality of the SOM?

**Quantization Error (QE ≈ 0.34):**
- Measures the average Euclidean distance between each training sample and its BMU's weight vector.
- A lower QE means the SOM's codebook (weight vectors) closely approximates the training data — better data compression and representation.
- Our QE of ~0.34 on [0,1]-scaled 22-dimensional data indicates the map reasonably represents the data without excessive distortion. In context, the theoretical minimum is 0 (perfect representation) and the maximum depends on data spread; our value is acceptable for a 100-neuron map covering 195 samples.
- If we had used a larger grid (e.g., 15×15), QE would decrease but at risk of memorizing rather than generalizing.

**Topographic Error (TE ≈ 0.067):**
- Measures topology preservation: the fraction of training samples for which the first and second BMU are **not adjacent** on the 2D grid.
- A TE near 0 means the map has learned a topology-preserving projection — similar data points land near each other on the grid.
- Our TE of ~0.067 means only ~6.7% of samples have their two closest neurons separated by more than one grid step — **excellent topology preservation**.
- This confirms that the SOM has successfully organized the Parkinson's voice data into a coherent 2D manifold where proximity on the grid reflects similarity in the 22-dimensional biomedical voice space.
